# ¿Está China acelerando su ritmo de lanzamientos espaciales?

En este proyecto analizo la evolución de los lanzamientos orbitales de China para comprobar si los [cuatro despegues realizados en 45 horas](https://www.space.com/space-exploration/launches-spacecraft/china-launches-4-different-rockets-to-orbit-in-45-hours-video) forman parte de una aceleración sostenida.

Después construiré un modelo de predicción para estimar con cuántos lanzamientos podría terminar China el año 2026.

## Conexión con la fuente de datos

Utilizo Launch Library 2, una base de datos pública que contiene los lanzamientos orbitales realizados desde 1957. Primero solicito una pequeña muestra de los lanzamientos efectuados desde China para comprobar la conexión y conocer la estructura de los datos.

In [1]:
# importo las librerías necesarias
import pandas as pd
import requests

# defino la dirección y los parámetros de la consulta
url = "https://ll.thespacedevs.com/2.3.0/launches/previous/"

parametros = {
    "country_code": "CHN",
    "limit": 5,
    "ordering": "-net",
    "mode": "list"
}

# solicito una pequeña muestra de lanzamientos chinos
respuesta = requests.get(
    url,
    params=parametros,
    timeout=30
)

# detengo la ejecución si la conexión ha fallado
respuesta.raise_for_status()

# convierto la respuesta en datos de python
datos = respuesta.json()

print("estado de la conexión:", respuesta.status_code)
print("lanzamientos encontrados:", datos["count"])

# convierto los primeros resultados en una tabla
muestra = pd.json_normalize(datos["results"])

display(muestra)

estado de la conexión: 200
lanzamientos encontrados: 7636


,id,url,name,response_mode,slug,launch_designator,last_updated,net,window_end,window_start,...,image.name,image.image_url,image.thumbnail_url,image.credit,image.license.id,image.license.name,image.license.priority,image.license.link,image.single_use,image.variants
0,5af31461-bce5-4cfb-a0ee-b527cf285d90,https://ll.thespacedevs.com/2.3.0/launches/5af...,Kinetica 1 | 9 satellites,list,kinetica-1-9-satellites-2,2026-220,2026-09-21T00:10:16Z,2026-09-20T04:03:00Z,2026-09-20T04:15:00Z,2026-09-20T03:54:00Z,...,[AUTO] Kinetica 1 - image,https://thespacedevs-prod.nyc3.digitaloceanspa...,https://thespacedevs-prod.nyc3.digitaloceanspa...,NaN,1,Unknown,9,NaN,True,[]
1,d1471f9d-e9d0-4146-8e97-90863e48bfc8,https://ll.thespacedevs.com/2.3.0/launches/d14...,Falcon 9 Block 5 | Starlink Group 15-27,list,falcon-9-block-5-starlink-group-15-27,2026-219,2026-09-20T23:59:42Z,2026-09-20T01:47:00Z,2026-09-20T05:47:00Z,2026-09-20T01:47:00Z,...,Starlink night fairing,https://thespacedevs-prod.nyc3.digitaloceanspa...,https://thespacedevs-prod.nyc3.digitaloceanspa...,SpaceX,5,CC BY-NC 2.0,1,https://creativecommons.org/licenses/by-nc/2.0/,False,[]
2,a527b585-f01f-4ecc-9230-711dcef72ef1,https://ll.thespacedevs.com/2.3.0/launches/a52...,Long March 2D | PIESAT-2 13-16,list,long-march-2d-piesat-2-13-16,2026-218,2026-09-20T23:46:44Z,2026-09-19T10:50:00Z,2026-09-19T11:06:00Z,2026-09-19T10:43:00Z,...,[AUTO] Long March 2D - image,https://thespacedevs-prod.nyc3.digitaloceanspa...,https://thespacedevs-prod.nyc3.digitaloceanspa...,NaN,1,Unknown,9,NaN,True,[]
3,f4c720ce-0cfd-43e6-b8dd-00273073bc01,https://ll.thespacedevs.com/2.3.0/launches/f4c...,Electron | Owl By The Dozen (StriX Launch 12),list,electron-owl-by-the-dozen-strix-launch-12,2026-217,2026-09-20T23:35:11Z,2026-09-19T03:22:00Z,2026-09-19T03:22:00Z,2026-09-19T03:22:00Z,...,[AUTO] Electron - image,https://thespacedevs-prod.nyc3.digitaloceanspa...,https://thespacedevs-prod.nyc3.digitaloceanspa...,NaN,1,Unknown,9,NaN,True,[]
4,ab3c1e6c-03fe-48a7-b6ce-30c79569bdfb,https://ll.thespacedevs.com/2.3.0/launches/ab3...,Soyuz 2.1b/Fregat | Glonass-K1 No. 20 (?),list,soyuz-21bfregat-glonass-k1-no-20,2026-216,2026-09-18T13:37:26Z,2026-09-17T16:50:00Z,2026-09-17T16:50:00Z,2026-09-17T16:50:00Z,...,Soyuz 2.1b on the pad,https://thespacedevs-prod.nyc3.digitaloceanspa...,https://thespacedevs-prod.nyc3.digitaloceanspa...,Roscosmos,14,Roscosmos Image Use Policy,3,https://www.roscosmos.ru/22650/,False,[]


### Comprobación del primer intento

La conexión con la API funciona correctamente, pero el parámetro utilizado inicialmente no filtra los lanzamientos por país. La consulta devolvió toda la base de datos.

Al solicitar una respuesta más completa comprobé que el país está asociado a la base de lanzamiento. Por eso, primero identificaré las ubicaciones chinas y después utilizaré sus identificadores para descargar solamente los lanzamientos necesarios.

In [2]:
# muestro los nombres de todas las columnas recibidas
for columna in muestra.columns:
    print(columna)

id
url
name
response_mode
slug
launch_designator
last_updated
net
window_end
window_start
infographic
status.id
status.name
status.abbrev
status.description
net_precision.id
net_precision.name
net_precision.abbrev
net_precision.description
image.id
image.name
image.image_url
image.thumbnail_url
image.credit
image.license.id
image.license.name
image.license.priority
image.license.link
image.single_use
image.variants


In [3]:
# localizo las columnas relacionadas con el país y el lugar de lanzamiento
palabras_clave = [
    "country",
    "location",
    "pad",
    "provider",
    "rocket",
    "status"
]

columnas_clave = [
    columna
    for columna in muestra.columns
    if any(
        palabra in columna.lower()
        for palabra in palabras_clave
    )
]

print("columnas relevantes:")
print(columnas_clave)

# reviso el contenido de las columnas encontradas
display(
    muestra[
        ["name", "net"] + columnas_clave
    ].head()
)

columnas relevantes:
['status.id', 'status.name', 'status.abbrev', 'status.description']


,name,net,status.id,status.name,status.abbrev,status.description
0,Kinetica 1 | 9 satellites,2026-09-20T04:03:00Z,3,Launch Successful,Success,The launch vehicle successfully inserted its p...
1,Falcon 9 Block 5 | Starlink Group 15-27,2026-09-20T01:47:00Z,3,Launch Successful,Success,The launch vehicle successfully inserted its p...
2,Long March 2D | PIESAT-2 13-16,2026-09-19T10:50:00Z,3,Launch Successful,Success,The launch vehicle successfully inserted its p...
3,Electron | Owl By The Dozen (StriX Launch 12),2026-09-19T03:22:00Z,3,Launch Successful,Success,The launch vehicle successfully inserted its p...
4,Soyuz 2.1b/Fregat | Glonass-K1 No. 20 (?),2026-09-17T16:50:00Z,3,Launch Successful,Success,The launch vehicle successfully inserted its p...


In [4]:
# solicito una muestra con información más completa
parametros_normales = {
    "limit": 5,
    "ordering": "-net",
    "mode": "normal"
}

respuesta_normal = requests.get(
    url,
    params=parametros_normales,
    timeout=30
)

respuesta_normal.raise_for_status()

datos_normales = respuesta_normal.json()

# convierto la nueva muestra en una tabla
muestra_normal = pd.json_normalize(
    datos_normales["results"]
)

# localizo las columnas que necesitaremos
palabras_clave = [
    "country",
    "location",
    "pad",
    "provider",
    "rocket",
    "status"
]

columnas_clave = [
    columna
    for columna in muestra_normal.columns
    if any(
        palabra in columna.lower()
        for palabra in palabras_clave
    )
]

print("número de columnas:", len(muestra_normal.columns))
print("\ncolumnas relevantes:")

for columna in columnas_clave:
    print(columna)

número de columnas: 163

columnas relevantes:
location_launch_attempt_count
pad_launch_attempt_count
location_launch_attempt_count_year
pad_launch_attempt_count_year
status.id
status.name
status.abbrev
status.description
launch_service_provider.response_mode
launch_service_provider.id
launch_service_provider.url
launch_service_provider.name
launch_service_provider.abbrev
launch_service_provider.type.id
launch_service_provider.type.name
rocket.id
rocket.configuration.response_mode
rocket.configuration.id
rocket.configuration.url
rocket.configuration.name
rocket.configuration.families
rocket.configuration.full_name
rocket.configuration.variant
pad.id
pad.url
pad.active
pad.agencies
pad.name
pad.image.id
pad.image.name
pad.image.image_url
pad.image.thumbnail_url
pad.image.credit
pad.image.license.id
pad.image.license.name
pad.image.license.priority
pad.image.license.link
pad.image.single_use
pad.image.variants
pad.description
pad.info_url
pad.wiki_url
pad.map_url
pad.latitude
pad.longitud

## Identificación de las bases espaciales chinas

Antes de descargar los lanzamientos, consulto qué bases y zonas de lanzamiento están registradas en China. Sus identificadores permitirán solicitar únicamente las misiones realizadas desde territorio chino.

In [5]:
# defino la dirección de las ubicaciones espaciales
url_ubicaciones = (
    "https://ll.thespacedevs.com/"
    "2.3.0/locations/"
)

# solicito todas las ubicaciones registradas en china
respuesta_ubicaciones = requests.get(
    url_ubicaciones,
    params={
        "country_code": "CHN",
        "limit": 100,
        "ordering": "name"
    },
    timeout=30
)

respuesta_ubicaciones.raise_for_status()

datos_ubicaciones = respuesta_ubicaciones.json()

# convierto las ubicaciones en una tabla
ubicaciones_china = pd.json_normalize(
    datos_ubicaciones["results"]
)

print(
    "ubicaciones encontradas:",
    datos_ubicaciones["count"]
)

display(
    ubicaciones_china[
        [
            "id",
            "name",
            "active",
            "total_launch_count"
        ]
    ]
)

ubicaciones encontradas: 5


,id,name,active,total_launch_count
0,185,Haiyang Oriental Spaceport,True,28
1,17,"Jiuquan Satellite Launch Center, People's Repu...",True,298
2,19,"Taiyuan Satellite Launch Center, People's Repu...",True,161
3,8,"Wenchang Space Launch Site, People's Republic ...",True,73
4,16,"Xichang Satellite Launch Center, People's Repu...",True,242


## Prueba del filtro por ubicaciones

La API registra cinco ubicaciones de lanzamiento activas en China. Utilizo sus identificadores para solicitar únicamente las misiones realizadas desde esas bases y comprobar que el filtro funciona correctamente.

In [6]:
# reúno los identificadores de las ubicaciones chinas
ids_ubicaciones = ",".join(
    ubicaciones_china["id"]
    .astype(str)
    .tolist()
)

print(
    "identificadores:",
    ids_ubicaciones
)

# solicito una muestra de lanzamientos desde china
parametros_china = {
    "location__ids": ids_ubicaciones,
    "limit": 5,
    "ordering": "-net",
    "mode": "normal"
}

respuesta_china = requests.get(
    url,
    params=parametros_china,
    timeout=30
)

respuesta_china.raise_for_status()

datos_china = respuesta_china.json()

muestra_china = pd.json_normalize(
    datos_china["results"]
)

print(
    "lanzamientos encontrados:",
    datos_china["count"]
)

display(
    muestra_china[
        [
            "name",
            "net",
            "status.name",
            "pad.location.name",
            "pad.country.name"
        ]
    ]
)

identificadores: 185,17,19,8,16
lanzamientos encontrados: 802


,name,net,status.name,pad.location.name,pad.country.name
0,Kinetica 1 | 9 satellites,2026-09-20T04:03:00Z,Launch Successful,"Jiuquan Satellite Launch Center, People's Repu...",China
1,Long March 2D | PIESAT-2 13-16,2026-09-19T10:50:00Z,Launch Successful,"Taiyuan Satellite Launch Center, People's Repu...",China
2,Kuaizhou 11 | SpaceTY-51 & 52,2026-09-17T02:40:00Z,Launch Successful,"Jiuquan Satellite Launch Center, People's Repu...",China
3,Long March 12 | SatNet LEO Group 25,2026-09-17T00:32:00Z,Launch Successful,"Wenchang Space Launch Site, People's Republic ...",China
4,Gravity-1 | SpaceSail Polar Group #16,2026-09-15T22:00:00Z,Launch Successful,Haiyang Oriental Spaceport,China


## Descarga de los lanzamientos chinos

Descargo todos los lanzamientos orbitales registrados en las cinco ubicaciones chinas. La API divide los resultados en varias páginas, por lo que recorro cada una hasta completar la descarga.

In [8]:
# continúo la descarga desde la página que quedó pendiente
while url_siguiente is not None:
    for intento in range(1, 4):
        try:
            respuesta_pagina = requests.get(
                url_siguiente,
                timeout=60
            )

            respuesta_pagina.raise_for_status()
            pagina = respuesta_pagina.json()
            break

        except requests.exceptions.RequestException as error:
            print(
                f"intento {intento} fallido:",
                type(error).__name__
            )

            if intento == 3:
                raise

            # espero unos segundos antes de volver a intentarlo
            time.sleep(5)

    # incorporo los nuevos registros
    registros_china.extend(
        pagina["results"]
    )

    print(
        "registros descargados:",
        len(registros_china)
    )

    url_siguiente = pagina["next"]

    # dejo una pausa entre consultas
    time.sleep(2)

# convierto la descarga completa en una tabla
lanzamientos_china = pd.json_normalize(
    registros_china
)

print(
    "\ndimensiones:",
    lanzamientos_china.shape
)

registros descargados: 600
registros descargados: 700
registros descargados: 800
registros descargados: 802

dimensiones: (802, 168)


## Selección y preparación de variables

La descarga contiene numerosos campos técnicos, imágenes y enlaces que no son necesarios para el análisis. Selecciono las variables relacionadas con la fecha, el resultado, el cohete, el operador y la ubicación del lanzamiento.

También creo variables temporales que permitirán analizar la evolución anual y mensual.

In [9]:
# selecciono las columnas necesarias
columnas_seleccionadas = {
    "id": "id",
    "name": "mision",
    "net": "fecha_hora",
    "status.name": "resultado",
    "status.abbrev": "resultado_abreviado",
    "launch_service_provider.name": "operador",
    "launch_service_provider.type.name": "tipo_operador",
    "rocket.configuration.full_name": "cohete",
    "pad.name": "plataforma",
    "pad.location.name": "base_lanzamiento",
    "pad.country.name": "pais",
    "pad.latitude": "latitud",
    "pad.longitude": "longitud"
}

# creo una tabla reducida y renombro sus columnas
df = (
    lanzamientos_china[
        list(columnas_seleccionadas.keys())
    ]
    .rename(columns=columnas_seleccionadas)
    .copy()
)

# convierto la fecha y la hora al formato temporal
df["fecha_hora"] = pd.to_datetime(
    df["fecha_hora"],
    utc=True,
    errors="coerce"
)

# creo las variables necesarias para el análisis temporal
df["fecha"] = df["fecha_hora"].dt.date
df["anio"] = df["fecha_hora"].dt.year
df["mes"] = df["fecha_hora"].dt.month
df["anio_mes"] = (
    df["fecha_hora"]
    .dt.to_period("M")
    .astype(str)
)

# ordeno los lanzamientos cronológicamente
df = (
    df.sort_values("fecha_hora")
    .reset_index(drop=True)
)

print("dimensiones:", df.shape)
print(
    "periodo:",
    df["fecha_hora"].min(),
    "-",
    df["fecha_hora"].max()
)

display(df.head())
display(df.tail())

dimensiones: (802, 17)
periodo: 1970-04-24 13:35:45+00:00 - 2026-09-20 04:03:00+00:00


C:\Users\rober\AppData\Local\Temp\ipykernel_34992\2445266607.py:40: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,id,mision,fecha_hora,resultado,resultado_abreviado,operador,tipo_operador,cohete,plataforma,base_lanzamiento,pais,latitud,longitud,fecha,anio,mes,anio_mes
0,738b8ac7-63c3-4216-9a7c-5e56edd113b6,Long March 1 | Dong Fang Hong 1,1970-04-24 13:35:45+00:00,Launch Successful,Success,Seventh Ministry of Machine Building Industry,Government,Long March 1,Launch Area 2A,"Jiuquan Satellite Launch Center, People's Repu...",China,41.308833,100.316512,1970-04-24,1970,4,1970-04
1,7cf1cd19-be35-4a63-8b4d-8821eb1dc03c,Long March 1 | Shijian 1,1971-03-03 12:02:00+00:00,Launch Successful,Success,Seventh Ministry of Machine Building Industry,Government,Long March 1,Launch Area 2A,"Jiuquan Satellite Launch Center, People's Repu...",China,41.308833,100.316512,1971-03-03,1971,3,1971-03
2,8e86cacb-19b5-4dc5-a505-2e7ab324d1ee,Feng Bao 1 | JSSW 1a,1973-09-18 12:12:00+00:00,Launch Failure,Failure,Seventh Ministry of Machine Building Industry,Government,Feng Bao 1,Launch Area 2B,"Jiuquan Satellite Launch Center, People's Repu...",China,41.306143,100.313229,1973-09-18,1973,9,1973-09
3,ba7b8eeb-feef-45ac-b884-204b2385f2e2,Feng Bao 1 | JSSW 1b,1974-07-12 13:55:00+00:00,Launch Failure,Failure,Seventh Ministry of Machine Building Industry,Government,Feng Bao 1,Launch Area 2B,"Jiuquan Satellite Launch Center, People's Repu...",China,41.306143,100.313229,1974-07-12,1974,7,1974-07
4,866ad4df-06e4-4103-9d9d-82ee079eb461,Long March 2 | FSW-0 0,1974-11-05 09:40:00+00:00,Launch Failure,Failure,Seventh Ministry of Machine Building Industry,Government,Long March 2A,Launch Area 2B,"Jiuquan Satellite Launch Center, People's Repu...",China,41.306143,100.313229,1974-11-05,1974,11,1974-11


,id,mision,fecha_hora,resultado,resultado_abreviado,operador,tipo_operador,cohete,plataforma,base_lanzamiento,pais,latitud,longitud,fecha,anio,mes,anio_mes
797,767f7827-53d8-4621-803b-f456a1e6b512,Gravity-1 | SpaceSail Polar Group #16,2026-09-15 22:00:00+00:00,Launch Successful,Success,Orienspace Technology,Commercial,Gravity-1,Yellow Sea (launch location 5),Haiyang Oriental Spaceport,China,31.200000,123.700000,2026-09-15,2026,9,2026-09
798,898f7df0-b6ce-4a5c-80b4-67e81842124e,Long March 12 | SatNet LEO Group 25,2026-09-17 00:32:00+00:00,Launch Successful,Success,China Aerospace Science and Technology Corpora...,Government,Long March 12,Commercial LC-2,"Wenchang Space Launch Site, People's Republic ...",China,19.597550,110.936481,2026-09-17,2026,9,2026-09
799,71aa4d56-0ce9-4c7d-a808-cc22f2ac5a84,Kuaizhou 11 | SpaceTY-51 & 52,2026-09-17 02:40:00+00:00,Launch Successful,Success,ExPace,Commercial,Kuaizhou 11,Launch Area 95A,"Jiuquan Satellite Launch Center, People's Repu...",China,40.969117,100.343333,2026-09-17,2026,9,2026-09
800,a527b585-f01f-4ecc-9230-711dcef72ef1,Long March 2D | PIESAT-2 13-16,2026-09-19 10:50:00+00:00,Launch Successful,Success,China Aerospace Science and Technology Corpora...,Government,Long March 2D,Launch Complex 9,"Taiyuan Satellite Launch Center, People's Repu...",China,38.863128,111.589567,2026-09-19,2026,9,2026-09
801,5af31461-bce5-4cfb-a0ee-b527cf285d90,Kinetica 1 | 9 satellites,2026-09-20 04:03:00+00:00,Launch Successful,Success,CAS Space,Commercial,Kinetica 1,Launch Area 130,"Jiuquan Satellite Launch Center, People's Repu...",China,40.818200,100.225140,2026-09-20,2026,9,2026-09


### Primera lectura de los datos

La base reúne 802 lanzamientos realizados desde ubicaciones chinas entre abril de 1970 y septiembre de 2026.

Los registros más recientes incluyen tanto operadores gubernamentales como empresas comerciales. Esto permitirá analizar no solo si ha aumentado el número de lanzamientos, sino también si el crecimiento reciente está relacionado con la aparición del sector espacial privado.

In [10]:
# creo una variable mensual sin generar avisos
df["anio_mes"] = df["fecha_hora"].dt.strftime("%Y-%m")

In [11]:
# cuento los valores ausentes de cada variable
print("valores ausentes:")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

# compruebo posibles registros duplicados
print(
    "identificadores duplicados:",
    df["id"].duplicated().sum()
)

# reviso los resultados de los lanzamientos
print("\nresultados registrados:")
display(
    df["resultado_abreviado"]
    .value_counts(dropna=False)
    .to_frame("lanzamientos")
)

# compruebo los países presentes
print("\npaíses incluidos:")
display(
    df["pais"]
    .value_counts(dropna=False)
    .to_frame("lanzamientos")
)

valores ausentes:


id                     0
mision                 0
fecha_hora             0
resultado              0
resultado_abreviado    0
operador               0
tipo_operador          0
cohete                 0
plataforma             0
base_lanzamiento       0
pais                   0
latitud                0
longitud               0
fecha                  0
anio                   0
mes                    0
anio_mes               0
dtype: int64

identificadores duplicados: 0

resultados registrados:


,lanzamientos
resultado_abreviado,
Success,758
Failure,38
Partial Failure,6



países incluidos:


,lanzamientos
pais,
China,802


### Calidad de la información

La base no contiene valores ausentes ni identificadores duplicados. Los 802 registros corresponden a lanzamientos realizados desde China.

De ellos, 758 terminaron con éxito, 38 fracasaron y 6 fueron éxitos parciales. Esto supone una tasa histórica de éxito del 94,5 %, aunque más adelante comprobaré cómo ha evolucionado con el tiempo.

In [12]:
# defino la ventana temporal mencionada en la noticia
inicio_racha = pd.Timestamp(
    "2026-09-15 00:00:00",
    tz="UTC"
)

fin_racha = pd.Timestamp(
    "2026-09-17 03:00:00",
    tz="UTC"
)

# selecciono los lanzamientos incluidos en la ventana
racha_2026 = df.loc[
    df["fecha_hora"].between(
        inicio_racha,
        fin_racha
    )
].copy()

# calculo las horas transcurridas entre cada lanzamiento
racha_2026["horas_desde_anterior"] = (
    racha_2026["fecha_hora"]
    .diff()
    .dt.total_seconds()
    .div(3600)
    .round(2)
)

# calculo la duración completa de la secuencia
duracion_racha = (
    racha_2026["fecha_hora"].max()
    - racha_2026["fecha_hora"].min()
).total_seconds() / 3600

print(
    "lanzamientos encontrados:",
    len(racha_2026)
)

print(
    "duración total:",
    round(duracion_racha, 2),
    "horas"
)

display(
    racha_2026[
        [
            "fecha_hora",
            "mision",
            "operador",
            "tipo_operador",
            "cohete",
            "base_lanzamiento",
            "resultado_abreviado",
            "horas_desde_anterior"
        ]
    ]
)


lanzamientos encontrados: 4
duración total: 44.23 horas


,fecha_hora,mision,operador,tipo_operador,cohete,base_lanzamiento,resultado_abreviado,horas_desde_anterior
796,2026-09-15 06:26:00+00:00,Zhuque-2E Block 2 | SpaceSail Polar Group #15,LandSpace,Commercial,Zhuque-2E Block 2,"Jiuquan Satellite Launch Center, People's Repu...",Success,NaN
797,2026-09-15 22:00:00+00:00,Gravity-1 | SpaceSail Polar Group #16,Orienspace Technology,Commercial,Gravity-1,Haiyang Oriental Spaceport,Success,15.57
798,2026-09-17 00:32:00+00:00,Long March 12 | SatNet LEO Group 25,China Aerospace Science and Technology Corpora...,Government,Long March 12,"Wenchang Space Launch Site, People's Republic ...",Success,26.53
799,2026-09-17 02:40:00+00:00,Kuaizhou 11 | SpaceTY-51 & 52,ExPace,Commercial,Kuaizhou 11,"Jiuquan Satellite Launch Center, People's Repu...",Success,2.13


### Resultado de la comprobación

Los datos confirman la noticia. China completó cuatro lanzamientos orbitales exitosos entre el 15 y el 17 de septiembre de 2026. Desde el despegue del Zhuque-2E hasta el del Kuaizhou 11 transcurrieron 44,23 horas, aproximadamente 44 horas y 14 minutos.

La secuencia también muestra la diversidad actual del sector espacial chino: participaron cuatro modelos de cohete y cuatro operadores diferentes.

La clasificación de la API considera comerciales a LandSpace, Orienspace y ExPace. Sin embargo, esta categoría describe su forma de operar y no implica necesariamente que sean empresas completamente privadas. ExPace está vinculada al sector estatal, por lo que esta diferencia deberá explicarse en el artículo.

In [13]:
# guardo una copia limpia de los datos
df.to_csv(
    "lanzamientos_china_limpios.csv",
    index=False,
    encoding="utf-8-sig"
)

print("archivo guardado correctamente")

archivo guardado correctamente


In [14]:
# resumo los lanzamientos y sus resultados por año
lanzamientos_anuales = (
    df.groupby("anio")
    .agg(
        lanzamientos=("id", "count"),
        exitos=(
            "resultado_abreviado",
            lambda valores: (
                valores == "Success"
            ).sum()
        ),
        fallos=(
            "resultado_abreviado",
            lambda valores: (
                valores == "Failure"
            ).sum()
        ),
        exitos_parciales=(
            "resultado_abreviado",
            lambda valores: (
                valores == "Partial Failure"
            ).sum()
        )
    )
    .reset_index()
)

# calculo la tasa anual de éxito
lanzamientos_anuales["tasa_exito"] = (
    lanzamientos_anuales["exitos"]
    .div(lanzamientos_anuales["lanzamientos"])
    .mul(100)
    .round(1)
)

# muestro la evolución desde el año 2000
display(
    lanzamientos_anuales.loc[
        lanzamientos_anuales["anio"] >= 2000
    ]
)

,anio,lanzamientos,exitos,fallos,exitos_parciales,tasa_exito
26,2000,5,5,0,0,100.0
27,2001,1,1,0,0,100.0
28,2002,5,4,1,0,80.0
29,2003,7,6,1,0,85.7
30,2004,8,8,0,0,100.0
31,2005,6,5,1,0,83.3
32,2006,6,6,0,0,100.0
33,2007,10,10,0,0,100.0
34,2008,11,11,0,0,100.0
35,2009,6,5,0,1,83.3


### Evolución anual

La actividad espacial china muestra un crecimiento evidente. Durante la primera década del siglo XXI, el país rara vez superaba los diez lanzamientos anuales. En 2018 alcanzó 39 y, desde 2021, encadena cifras superiores a 50.

El máximo de la serie se produjo en 2025, con 93 lanzamientos, casi cinco veces los 19 registrados en 2015. Este aumento no parece deberse únicamente a un año excepcional, sino a una tendencia mantenida durante la última década.

Además, el crecimiento de la actividad no ha provocado un deterioro evidente de la fiabilidad: en los últimos años, la tasa de éxito se ha mantenido generalmente por encima del 90 %.

In [15]:
# creo el día del mes para aplicar el mismo corte temporal
df["dia"] = df["fecha_hora"].dt.day

# selecciono los lanzamientos realizados hasta el 20 de septiembre
df_hasta_corte = df.loc[
    (df["mes"] < 9)
    |
    (
        (df["mes"] == 9)
        & (df["dia"] <= 20)
    )
].copy()

# cuento los lanzamientos acumulados hasta esa fecha
comparacion_corte = (
    df_hasta_corte.groupby("anio")
    .agg(
        lanzamientos_hasta_20_septiembre=(
            "id",
            "count"
        )
    )
    .reset_index()
)

# calculo el crecimiento respecto al año anterior
comparacion_corte["variacion_anual"] = (
    comparacion_corte[
        "lanzamientos_hasta_20_septiembre"
    ]
    .pct_change()
    .mul(100)
    .round(1)
)

display(
    comparacion_corte.loc[
        comparacion_corte["anio"] >= 2010
    ]
)

,anio,lanzamientos_hasta_20_septiembre,variacion_anual
33,2010,8,166.7
34,2011,10,25.0
35,2012,12,20.0
36,2013,6,-50.0
37,2014,5,-16.7
38,2015,7,40.0
39,2016,14,100.0
40,2017,8,-42.9
41,2018,25,212.5
42,2019,18,-28.0


### El ritmo de 2026

La comparación con el mismo periodo de años anteriores confirma que la aceleración continúa en 2026.

Hasta el 20 de septiembre, China había realizado 68 lanzamientos, frente a los 56 registrados en la misma fecha de 2025. Esto supone un crecimiento interanual del 21,4 %.

Además, antes de terminar septiembre, China ya había igualado todos los lanzamientos realizados durante 2024 y superado el total de 2023. Por tanto, los cuatro despegues en 45 horas no son un episodio aislado: se producen dentro del año de mayor actividad de toda la serie.

In [16]:
# calculo el tiempo entre lanzamientos dentro de cada año
df["dias_desde_anterior"] = (
    df.groupby("anio")["fecha_hora"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

# resumo la cadencia anual de lanzamientos
cadencia_anual = (
    df.groupby("anio")
    .agg(
        lanzamientos=("id", "count"),
        espera_media=(
            "dias_desde_anterior",
            "mean"
        ),
        espera_mediana=(
            "dias_desde_anterior",
            "median"
        ),
        intervalo_minimo=(
            "dias_desde_anterior",
            "min"
        ),
        intervalos_hasta_2_dias=(
            "dias_desde_anterior",
            lambda valores: (
                valores <= 2
            ).sum()
        )
    )
    .reset_index()
)

# redondeo las medidas temporales
columnas_redondeo = [
    "espera_media",
    "espera_mediana",
    "intervalo_minimo"
]

cadencia_anual[columnas_redondeo] = (
    cadencia_anual[columnas_redondeo]
    .round(2)
)

display(
    cadencia_anual.loc[
        cadencia_anual["anio"] >= 2010
    ]
)

,anio,lanzamientos,espera_media,espera_mediana,intervalo_minimo,intervalos_hasta_2_dias
36,2010,15,23.94,15.89,4.09,0
37,2011,19,14.24,10.32,2.10,0
38,2012,19,19.14,15.67,2.25,0
39,2013,15,17.04,9.48,2.06,0
40,2014,16,18.33,9.90,0.98,1
41,2015,19,15.17,8.94,1.54,1
42,2016,22,16.54,13.51,1.98,1
43,2017,18,20.83,9.29,2.65,0
44,2018,39,9.32,8.84,0.59,6
45,2019,34,10.63,8.94,0.12,4


### Cada vez pasa menos tiempo entre lanzamientos

El crecimiento también aparece al medir la separación entre misiones. En 2010, la espera mediana entre dos lanzamientos chinos era de casi 16 días. En 2026 se ha reducido a 2,4 días.

Dicho de otra manera, el ritmo habitual de lanzamiento es ahora aproximadamente seis veces más rápido que en 2010.

Los periodos de actividad muy concentrada también son cada vez más frecuentes. Hasta septiembre de 2026 se habían registrado 30 ocasiones en las que dos lanzamientos estuvieron separados por un máximo de 48 horas, frente a ninguna en 2010.

La secuencia de cuatro cohetes en 45 horas es excepcional, pero encaja en una tendencia sostenida hacia lanzamientos cada vez más frecuentes.

In [17]:
# cuento los lanzamientos por año y tipo de operador
actividad_por_tipo = pd.crosstab(
    df["anio"],
    df["tipo_operador"]
)

# calculo el total anual
actividad_por_tipo["total"] = (
    actividad_por_tipo.sum(axis=1)
)

# calculo el peso de los operadores comerciales
if "Commercial" in actividad_por_tipo.columns:
    actividad_por_tipo[
        "porcentaje_comercial"
    ] = (
        actividad_por_tipo["Commercial"]
        .div(actividad_por_tipo["total"])
        .mul(100)
        .round(1)
    )

# convierto el año en una columna
actividad_por_tipo = (
    actividad_por_tipo
    .reset_index()
)

display(
    actividad_por_tipo.loc[
        actividad_por_tipo["anio"] >= 2010
    ]
)

tipo_operador,anio,Commercial,Government,total,porcentaje_comercial
36,2010,0,15,15,0.0
37,2011,0,19,19,0.0
38,2012,0,19,19,0.0
39,2013,0,15,15,0.0
40,2014,0,16,16,0.0
41,2015,0,19,19,0.0
42,2016,0,22,22,0.0
43,2017,1,17,18,5.6
44,2018,2,37,39,5.1
45,2019,8,26,34,23.5


### El auge de los operadores comerciales

Hasta 2016, todos los lanzamientos de la base aparecen asociados a operadores gubernamentales. Los primeros registros comerciales llegan en 2017 y su presencia aumenta con rapidez durante los años siguientes.

En 2026, los operadores clasificados como comerciales habían realizado 23 lanzamientos hasta el 20 de septiembre, el 33,8 % de toda la actividad del año. Es la proporción más alta de la serie.

El crecimiento no se explica solamente por estas compañías, porque los lanzamientos gubernamentales también han aumentado. Sin embargo, la entrada de nuevos operadores está ampliando la capacidad de China para realizar varias misiones en periodos cada vez más cortos.

La categoría comercial no equivale siempre a propiedad completamente privada: algunas empresas mantienen vínculos con organismos estatales.

In [18]:
# ordeno las fechas para analizar ventanas de 45 horas
fechas_ordenadas = (
    df["fecha_hora"]
    .sort_values()
    .reset_index(drop=True)
)

ventanas_45h = []

# cuento cuántos lanzamientos siguen a cada despegue
for posicion, inicio in enumerate(fechas_ordenadas):
    limite = inicio + pd.Timedelta(hours=45)

    cantidad = (
        fechas_ordenadas
        .between(inicio, limite)
        .sum()
    )

    ventanas_45h.append(
        {
            "inicio": inicio,
            "fin": limite,
            "lanzamientos": cantidad
        }
    )

ventanas_45h = pd.DataFrame(
    ventanas_45h
)

maximo_45h = ventanas_45h[
    "lanzamientos"
].max()

print(
    "máximo histórico en 45 horas:",
    maximo_45h
)

# muestro las ventanas con cuatro o más lanzamientos
display(
    ventanas_45h.loc[
        ventanas_45h["lanzamientos"] >= 4
    ]
    .sort_values(
        ["lanzamientos", "inicio"],
        ascending=[False, False]
    )
)

máximo histórico en 45 horas: 4


,inicio,fin,lanzamientos
796,2026-09-15 06:26:00+00:00,2026-09-17 03:26:00+00:00,4
721,2025-12-08 22:11:00+00:00,2025-12-10 19:11:00+00:00,4
710,2025-11-08 21:01:00+00:00,2025-11-10 18:01:00+00:00,4


### Un récord igualado, no un caso sin precedentes

Los cuatro lanzamientos de septiembre de 2026 igualan la mayor concentración de misiones registrada en toda la serie: nunca aparecen más de cuatro despegues chinos dentro de una ventana de 45 horas.

Sin embargo, no es la primera vez que ocurre. La misma concentración ya se había registrado dos veces a finales de 2025, en noviembre y diciembre.

Este resultado refuerza la idea de una aceleración sostenida. Lo extraordinario no es solamente que China pueda lanzar cuatro cohetes en menos de dos días, sino que haya alcanzado ese ritmo tres veces en menos de un año.

In [19]:
# selecciono las ventanas que igualan el récord
ventanas_record = ventanas_45h.loc[
    ventanas_45h["lanzamientos"] == maximo_45h
].copy()

resumen_rachas = []

# recupero los lanzamientos incluidos en cada racha
for _, ventana in ventanas_record.iterrows():
    lanzamientos_racha = df.loc[
        df["fecha_hora"].between(
            ventana["inicio"],
            ventana["fin"]
        )
    ].copy()

    duracion_real = (
        lanzamientos_racha["fecha_hora"].max()
        - lanzamientos_racha["fecha_hora"].min()
    ).total_seconds() / 3600

    resumen_rachas.append(
        {
            "inicio": lanzamientos_racha[
                "fecha_hora"
            ].min(),
            "fin_real": lanzamientos_racha[
                "fecha_hora"
            ].max(),
            "duracion_horas": round(
                duracion_real,
                2
            ),
            "cohetes": " | ".join(
                lanzamientos_racha["cohete"]
            )
        }
    )

resumen_rachas = pd.DataFrame(
    resumen_rachas
)

display(resumen_rachas)

,inicio,fin_real,duracion_horas,cohetes
0,2025-11-08 21:01:00+00:00,2025-11-10 04:02:53+00:00,31.03,Long March 11 | Kinetica 1 | Long March 12 | C...
1,2025-12-08 22:11:00+00:00,2025-12-10 04:03:00+00:00,29.87,Long March 6A | Long March 4B | Long March 3B/...
2,2026-09-15 06:26:00+00:00,2026-09-17 02:40:00+00:00,44.23,Zhuque-2E Block 2 | Gravity-1 | Long March 12 ...


### China ya lo había hecho más rápido

La secuencia de septiembre de 2026 iguala el máximo de cuatro lanzamientos, pero es la más lenta de las tres rachas récord.

En noviembre de 2025, China completó cuatro misiones en 31 horas. Un mes después volvió a hacerlo en menos de 30 horas. La secuencia de 2026 necesitó algo más de 44 horas.

Por tanto, el titular de los cuatro cohetes en 45 horas describe un ritmo extraordinario, pero no un nuevo récord. Los datos revelan algo más relevante: este nivel de actividad ha comenzado a repetirse.

In [1]:
# importo las librerías necesarias
import pandas as pd
import plotly.express as px

# recupero la tabla limpia guardada
df = pd.read_csv(
    "lanzamientos_china_limpios.csv",
    parse_dates=["fecha_hora"]
)

# reconstruyo la tabla utilizada por el gráfico
actividad_por_tipo = pd.crosstab(
    df["anio"],
    df["tipo_operador"]
)

actividad_por_tipo["total"] = (
    actividad_por_tipo.sum(axis=1)
)

actividad_por_tipo[
    "porcentaje_comercial"
] = (
    actividad_por_tipo["Commercial"]
    .div(actividad_por_tipo["total"])
    .mul(100)
    .round(1)
)

actividad_por_tipo = (
    actividad_por_tipo.reset_index()
)

print("datos recuperados:", df.shape)

datos recuperados: (802, 17)


In [3]:
# importo la librería para crear gráficos interactivos
import plotly.express as px

# preparo los datos desde 2010
datos_grafico_anual = (
    actividad_por_tipo.loc[
        actividad_por_tipo["anio"] >= 2010,
        [
            "anio",
            "Government",
            "Commercial"
        ]
    ]
    .melt(
        id_vars="anio",
        var_name="tipo",
        value_name="lanzamientos"
    )
)

# traduzco las categorías para facilitar la lectura
datos_grafico_anual["tipo"] = (
    datos_grafico_anual["tipo"]
    .replace(
        {
            "Government": "operadores gubernamentales",
            "Commercial": "operadores comerciales"
        }
    )
)

# creo el gráfico de barras apiladas
fig_anual = px.bar(
    datos_grafico_anual,
    x="anio",
    y="lanzamientos",
    color="tipo",
    title="China multiplica su ritmo de lanzamientos",
    labels={
        "anio": "año",
        "lanzamientos": "lanzamientos",
        "tipo": "tipo de operador"
    },
    color_discrete_map={
        "operadores gubernamentales": "#bc2a2a",
        "operadores comerciales": "#f0a44b"
    }
)

# indico que 2026 todavía no ha terminado
fig_anual.add_annotation(
    x=2026,
    y=70,
    text="hasta el 20 de septiembre",
    showarrow=True,
    arrowhead=2,
    ax=-90,
    ay=-45
)

fig_anual.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text=""
)

fig_anual.show()

# guardo una versión interactiva independiente
fig_anual.write_html(
    "lanzamientos_china_por_anio.html",
    include_plotlyjs="cdn"
)

In [4]:
# ordeno los lanzamientos cronológicamente
df = df.sort_values(
    "fecha_hora"
).reset_index(drop=True)

# calculo los días entre lanzamientos del mismo año
df["dias_desde_anterior"] = (
    df.groupby("anio")["fecha_hora"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

# calculo la espera mediana de cada año
cadencia_anual = (
    df.groupby("anio")
    .agg(
        espera_mediana=(
            "dias_desde_anterior",
            "median"
        )
    )
    .reset_index()
)

# selecciono el periodo más relevante
cadencia_grafico = cadencia_anual.loc[
    cadencia_anual["anio"] >= 2010
].copy()

# creo el gráfico interactivo
fig_cadencia = px.line(
    cadencia_grafico,
    x="anio",
    y="espera_mediana",
    markers=True,
    title=(
        "China reduce el tiempo entre "
        "sus lanzamientos"
    ),
    labels={
        "anio": "año",
        "espera_mediana": (
            "días habituales entre lanzamientos"
        )
    }
)

fig_cadencia.update_traces(
    line={
        "color": "#bc2a2a",
        "width": 3
    },
    marker={
        "size": 8,
        "color": "#f0a44b"
    }
)

fig_cadencia.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig_cadencia.show()

# guardo el gráfico interactivo
fig_cadencia.write_html(
    "cadencia_lanzamientos_china.html",
    include_plotlyjs="cdn"
)

### La aceleración también se observa en la cadencia

La frecuencia anual puede variar, pero la tendencia general es clara. A comienzos de la década de 2010, la espera habitual entre dos misiones se situaba entre diez y dieciséis días.

Desde 2021, esa separación cae por debajo de cinco días y alcanza su mínimo en 2026, con una mediana de 2,4 días. China no solo lanza más cohetes al año: también ha adquirido la capacidad de encadenarlos con mucha más rapidez.

## Conclusiones del análisis exploratorio

### China está acelerando realmente su actividad espacial

Los datos confirman que el aumento de los lanzamientos chinos no responde a un episodio aislado. Se trata de una transformación sostenida que se aprecia tanto en el número de misiones como en la frecuencia con la que se realizan.

En 2015, China completó 19 lanzamientos orbitales. Diez años después alcanzó los 93, casi cinco veces más. El crecimiento tampoco se concentra únicamente en el último año: desde 2021, el país ha superado sucesivamente los 50 lanzamientos anuales y ha ido elevando su máximo histórico.

Por tanto, los cuatro cohetes lanzados en 45 horas no aparecen de forma repentina. Son la consecuencia visible de una capacidad espacial que lleva varios años creciendo.

### 2026 avanza más rápido que el año del récord

El total de 2026 todavía no puede compararse directamente con el de años completos. Para evitar una conclusión engañosa, he medido cuántos lanzamientos llevaba China hasta el 20 de septiembre de cada año.

En esa fecha de 2025 acumulaba 56 misiones. En 2026 ya había alcanzado 68, lo que representa un incremento del 21,4 %.

La comparación resulta todavía más significativa al observar los años anteriores: antes de terminar septiembre, China ya había igualado los 68 lanzamientos de todo 2024 y superado los 67 realizados durante 2023.

Esto permite afirmar que 2026 no solo mantiene el crecimiento, sino que avanza a un ritmo superior al de 2025, el año que actualmente conserva el récord con 93 lanzamientos.

### El cambio más importante está en el tiempo entre misiones

El número anual de lanzamientos muestra el crecimiento general, pero la reducción del tiempo entre despegues permite entender mejor la magnitud del cambio.

En 2010, la espera mediana entre dos lanzamientos chinos era de 15,9 días. En 2025 se redujo hasta 2,6 días y, en 2026, descendió nuevamente hasta 2,4 días.

Esto significa que el ritmo habitual es ahora aproximadamente seis veces más rápido que a comienzos de la década pasada.

También se han vuelto más frecuentes las misiones separadas por periodos muy cortos. En 2010 no hubo ningún par de lanzamientos con menos de dos días de diferencia. Hasta el 20 de septiembre de 2026 ya se habían producido 30 intervalos de este tipo, casi tantos como los 33 registrados durante todo 2025.

China no solo lanza más cohetes: ha desarrollado la capacidad operativa necesaria para preparar y ejecutar misiones simultáneamente desde diferentes bases.

### Las 45 horas de la noticia no fueron un nuevo récord

La noticia queda confirmada por los datos. Entre el lanzamiento del Zhuque-2E del 15 de septiembre y el del Kuaizhou 11 del día 17 transcurrieron 44,23 horas, aproximadamente 44 horas y 14 minutos. Los cuatro cohetes alcanzaron la órbita con éxito.

La secuencia iguala el máximo histórico de cuatro lanzamientos chinos dentro de una ventana de 45 horas. Sin embargo, no fue la más rápida.

En noviembre de 2025, China ya había completado cuatro misiones en 31 horas. En diciembre del mismo año volvió a hacerlo en solo 29 horas y 52 minutos.

El titular de las 45 horas es correcto y describe una concentración de lanzamientos extraordinaria, pero los datos aportan un contexto diferente: China ya lo había conseguido dos veces y en menos tiempo.

El verdadero dato noticioso no es, por tanto, un nuevo récord. Es que esta capacidad se ha repetido tres veces en menos de un año y empieza a parecer menos excepcional.

### Los operadores comerciales están ampliando la capacidad de lanzamiento

Hasta 2016, todas las misiones de la base aparecen asociadas a operadores gubernamentales. La actividad comercial comienza a aparecer en 2017 y gana importancia durante los años siguientes.

En 2023, los operadores clasificados como comerciales realizaron 20 lanzamientos. Hasta septiembre de 2026 ya acumulaban 23 y representaban el 33,8 % de la actividad del año, el porcentaje más alto de toda la serie.

Este crecimiento no sustituye al programa estatal. Los operadores gubernamentales también han aumentado considerablemente su número de misiones. La aceleración procede de la suma de ambos: una infraestructura pública cada vez más activa y un ecosistema comercial que añade cohetes, bases y capacidad operativa.

Esta clasificación debe interpretarse con cautela. Que un operador figure como comercial no significa necesariamente que sea completamente privado. Algunas compañías mantienen vínculos empresariales o institucionales con el Estado chino.

### Más lanzamientos no han supuesto muchos más fallos

La aceleración podría haber provocado una pérdida de fiabilidad, pero los datos no muestran un deterioro evidente.

De los 802 lanzamientos analizados, 758 terminaron con éxito, 38 fracasaron y 6 obtuvieron un éxito parcial. La tasa histórica de éxito es del 94,5 %.

En 2025, el año con más actividad, la tasa alcanzó el 96,8 %. En 2026 se sitúa en el 94,1 %. Aunque los fallos siguen existiendo, el fuerte aumento del número de misiones no ha venido acompañado de una caída proporcional de la fiabilidad.

### Qué permiten afirmar los datos

El análisis confirma tres cambios:

1. China realiza muchos más lanzamientos que hace una década.
2. El tiempo habitual entre misiones se ha reducido de forma drástica.
3. Las concentraciones de varios lanzamientos en uno o dos días se están volviendo más frecuentes.

Por eso, la respuesta a la pregunta inicial es clara: **China está acelerando realmente su ritmo de lanzamientos orbitales**.

La secuencia de septiembre de 2026 no fue un récord de velocidad, pero sí otra demostración de que el país puede mantener varias campañas de lanzamiento activas al mismo tiempo.

### Limitaciones del análisis

La base utilizada recoge los lanzamientos efectuados desde ubicaciones chinas. El análisis mide la frecuencia de las misiones, pero no su tamaño ni su dificultad.

Un lanzamiento pequeño y otro capaz de transportar una carga mucho mayor cuentan de la misma manera. Tampoco se han incorporado la masa situada en órbita, el coste de las misiones, el número de satélites, su finalidad o el grado de reutilización de los cohetes.

Por tanto, el número de lanzamientos permite medir la intensidad de la actividad, pero no basta por sí solo para determinar qué país posee la mayor capacidad espacial.

Además, los datos de 2026 terminan el 20 de septiembre. El total definitivo todavía no se conoce y será precisamente el objetivo del siguiente análisis: estimar con cuántos lanzamientos podría cerrar China el año.